In [11]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from pydoe import lhs

In [13]:
device = torch.device("cpu")

In [14]:
def weights_init(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight.data)
        torch.nn.init.zeros_(m.bias.data)

In [15]:
class DNN(nn.Module):
    """Fully connected deep neural network with min-max input normalization.

    Used as the surrogate model in a PINN. Inputs are normalized to [0, 1]
    before passing through the network. The output layer has no activation
    so the network can predict unbounded values.

    Args:
        in_dim: Number of input features (e.g. 2 for x and t).
        hidden_dim: Width of each hidden layer.
        out_dim: Number of output features (e.g. 1 for u).
        depth: Number of hidden layers.
        min_: Per-feature minimum values for input normalization.
        max_: Per-feature maximum values for input normalization.
        activation: Activation function applied after each hidden layer.
    """

    def __init__(self, in_dim, hidden_dim, out_dim, depth: int, min_, max_, activation=nn.Tanh()) -> None:
        super().__init__()
        self.in_layer = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList()
        for _ in range(depth):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.out_layer = nn.Linear(hidden_dim, out_dim)
        self.activation = activation
        self.min = min_
        self.max = max_

    def forward(self, x):
        """Forward pass: normalize input, pass through layers, return prediction."""
        x = (x - self.min) / (self.max - self.min)
        x = self.activation((self.in_layer(x)))
        for layer in self.layers:
            x = self.activation((layer(x)))
        x = self.out_layer(x)
        return x

In [18]:
class PINN():
    def __init__(self) -> None:
        self.iter = 0
        self.loss_dict = {'iters': [], 'loss': []}
        Nf = 10000
        Nb = 100

        xt_domain = torch.tensor([[-1, 0], [1, 1]])
        min_ = xt_domain[0].to(device)
        max_ = xt_domain[1].to(device)
        self.xt_in = (xt_domain[1] - xt_domain[0]) * torch.tensor(lhs(2, Nf), dtype=torch.float) + xt_domain[0]

        x_init = np.random.uniform(-1, 1, Nb)
        t_init = np.zeros_like(x_init)
        xt_init = np.stack([x_init, t_init], axis=1)

        t_bc1 = np.random.uniform(0, 1, Nb)
        x_bc1 = np.ones_like(t_bc1)
        xt_bc1 = np.stack([x_bc1, t_bc1], axis=1)

        t_bc2 = np.random.uniform(0, 1, Nb)
        x_bc2 = -1 * np.ones_like(t_bc2)
        xt_bc2 = np.stack([x_bc2, t_bc2], axis=1)

        self.xt_bc1 = torch.tensor(xt_bc1, dtype=torch.float)
        self.xt_bc2 = torch.tensor(xt_bc2, dtype=torch.float)
        self.xt_init = torch.tensor(xt_init, dtype=torch.float)

        self.xt_in = self.xt_in.to(device)
        self.xt_in.requires_grad_(True)
        self.xt_bc1 = self.xt_bc1.to(device)
        self.xt_bc2 = self.xt_bc2.to(device)
        self.xt_init = self.xt_init.to(device)

        self.net = DNN(2, 20, 1, depth=5, min_=min_, max_=max_).to(device)
        self.net.apply(weights_init)
        self.lossfn = torch.nn.MSELoss().to(device)

    def loss_compute(self):
        self.iter += 1

        u = self.net(self.xt_in)
        grads = torch.autograd.grad(u.sum(), self.xt_in, create_graph=True)[0]
        u_t = grads[:, 1]
        u_x = grads[:, 0]
        u_xx = torch.autograd.grad(u_x.sum(), self.xt_in, create_graph=True)[0][:, 0]
        res = u_t + u.squeeze() * u_x - 0.01 / torch.pi * u_xx
        loss_pde = self.lossfn(res, torch.zeros_like(res))

        u_init_exact = -torch.sin(torch.pi * self.xt_init[:, 0])
        u_init_pred = self.net(self.xt_init)
        loss_ic = self.lossfn(u_init_exact, u_init_pred.squeeze())

        u_bc1 = self.net(self.xt_bc1)
        u_bc2 = self.net(self.xt_bc2)
        u_bc = torch.cat([u_bc1, u_bc2, u_bc1 - u_bc2], dim=1)
        loss_bc = self.lossfn(u_bc, torch.zeros_like(u_bc))

        loss = +loss_pde + loss_ic + loss_bc
        self.loss_dict['iters'].append(self.iter)
        self.loss_dict['loss'].append(loss.item())
        return loss

In [19]:
pinn = PINN()
pinn

In [ ]:
# adam = torch.optim.Adam(pinn.net.parameters(), lr=0.01)
lbfgs = torch.optim.LBFGS(
    pinn.net.parameters(),
    lr=1.0,
    max_iter=50000,
    max_eval=50000,
    history_size=50,
    tolerance_grad=1e-5,
    tolerance_change=1.0 * np.finfo(float).eps,
    line_search_fn="strong_wolfe",  # better numerical stability
    )

In [21]:
def closure():
    lbfgs.zero_grad()
    loss = pinn.loss_compute()
    loss.backward()
    print(f"\riter : {pinn.iter}     loss : {loss.item():.3e} ", end="")
    if pinn.iter % 300 == 0:
        print("")
    return loss

In [22]:
lbfgs.step(closure)

iter : 300     loss : 9.336e-03 
iter : 600     loss : 8.995e-04 
iter : 900     loss : 3.139e-04 
iter : 1200     loss : 1.386e-04 
iter : 1500     loss : 8.404e-05 
iter : 1800     loss : 5.557e-05 
iter : 2100     loss : 3.712e-05 
iter : 2400     loss : 2.777e-05 
iter : 2700     loss : 2.006e-05 
iter : 3000     loss : 1.600e-05 
iter : 3300     loss : 1.296e-05 
iter : 3600     loss : 1.044e-05 
iter : 3900     loss : 8.695e-06 
iter : 4200     loss : 7.145e-06 
iter : 4500     loss : 5.909e-06 
iter : 4800     loss : 4.942e-06 
iter : 5100     loss : 4.243e-06 
iter : 5317     loss : 3.974e-06 

tensor(0.6216, grad_fn=<AddBackward0>)